In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

APPLICATION_ID = os.environ["EMR_APPLICATION_ID"]

# As of July 16, 2026, Terraform does not yet support the session_enabled parameter.
!aws emr-serverless update-application \
  --application-id {APPLICATION_ID} \
  --interactive-configuration \
        sessionEnabled=true


aws: [ERROR]: An error occurred (ValidationException) when calling the UpdateApplication operation: Only the following configurations can be updated when an application is in STARTED state: schedulerConfiguration.maxConcurrentRuns, extendedSupportConfiguration, imageConfiguration, workerTypeSpecifications.imageConfiguration, maximumCapacity.


In [13]:
import os
import time
import boto3
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

load_dotenv()

APPLICATION_ID = os.environ["EMR_APPLICATION_ID"]
EXECUTION_ROLE = os.environ["EMR_EXECUTION_ROLE_ARN"]
REGION = os.environ.get("AWS_REGION", "ap-southeast-1")

client = boto3.client('emr-serverless', region_name=REGION)

# Start the session
response = client.start_session(
    applicationId=APPLICATION_ID,
    executionRoleArn=EXECUTION_ROLE
)
session_id = response['sessionId']
print(f"Session {session_id} starting...")

# Wait for the session to be ready
while True:
    response = client.get_session(
        applicationId=APPLICATION_ID,
        sessionId=session_id
    )
    state = response['session']['state']
    print(f"Session state: {state}")
    if state in ('STARTED', 'IDLE'):
        break
    if state in ('FAILED', 'TERMINATED'):
        raise Exception(f"Session failed: {response['session'].get('stateDetails', 'Unknown error')}")
    time.sleep(5)

# Retrieve the Spark Connect endpoint and authentication token
response = client.get_session_endpoint(
    applicationId=APPLICATION_ID,
    sessionId=session_id
)
auth_token = response['authToken']
endpoint_url = response['endpoint']
connect_url = endpoint_url.replace("https://", "sc://", 1) + ":443/;use_ssl=true;"
connect_url += f"x-aws-proxy-auth={auth_token}"

os.environ['SPARK_REMOTE'] = connect_url

# Ghi ra file (gitignored) để chạy dbt ở terminal ngoài cũng trỏ vào cùng
# session này: eval "$(cat ../.spark_remote_env)" rồi dbt debug / dbt run.
# Không commit file này — nó chứa authToken thật, hết hạn theo TTL của session.
with open("../.spark_remote_env", "w") as f:
    f.write(f"export SPARK_REMOTE='{connect_url}'\n")

print("SPARK_REMOTE set for this notebook session, and written to ../.spark_remote_env")
print(f"# terminate when done: client.terminate_session(applicationId='{APPLICATION_ID}', sessionId='{session_id}')")

Session 00g78j0vi2gn3q26 starting...
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTING
Session state: STARTED
SPARK_REMOTE set for this notebook session, and written to ../.spark_remote_env
# terminate when done: client.terminate_session(applicationId='00g6f3nethsue025', sessionId='00g78j0vi2gn3q26')


In [15]:
# Quick sanity check that the remote Spark Connect session works
spark = SparkSession.builder.remote(os.environ['SPARK_REMOTE']).getOrCreate()
print(f"Connected. Spark version: {spark.version}")

spark.sql("SELECT 1+1 AS result").show()

df = spark.range(100).withColumn("squared", col("id") * col("id"))
df.show(10)
print(f"Count: {df.count()}")

spark.stop()

Connected. Spark version: 3.5.6-amzn-2
+------+
|result|
+------+
|     2|
+------+

+---+-------+
| id|squared|
+---+-------+
|  0|      0|
|  1|      1|
|  2|      4|
|  3|      9|
|  4|     16|
|  5|     25|
|  6|     36|
|  7|     49|
|  8|     64|
|  9|     81|
+---+-------+
only showing top 10 rows

Count: 100


In [16]:
# Test dbt thật sự qua remote Spark Connect session (subprocess kế thừa
# SPARK_REMOTE từ os.environ đã set ở cell trên).
import subprocess

def run_dbt(*args):
    result = subprocess.run(
        ["dbt", *args, "--project-dir", "../dbt", "--profiles-dir", os.path.expanduser("~/.dbt")],
        env=os.environ,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    return result.returncode

run_dbt("debug")

09:19:30  Running with dbt=1.11.12
09:19:30  dbt version: 1.11.12
09:19:30  python version: 3.12.1
09:19:30  python path: /home/codespace/.cache/pypoetry/virtualenvs/aws-pipeline-W5ufifAU-py3.12/bin/python
09:19:30  os info: Linux-6.8.0-1052-azure-x86_64-with-glibc2.39
09:19:31  Using profiles dir at /home/codespace/.dbt
09:19:31  Using profiles.yml file at /home/codespace/.dbt/profiles.yml
09:19:31  Using dbt_project.yml file at ../dbt/dbt_project.yml
09:19:31  adapter type: spark
09:19:31  adapter version: 1.10.3
09:19:31  Configuration:
09:19:31    profiles.yml file [OK found and valid]
09:19:31    dbt_project.yml file [OK found and valid]
09:19:31  Required dependencies:
09:19:31   - git [OK found]

09:19:31  Connection:
09:19:31    host: NA
09:19:31    port: 443
09:19:31    cluster: None
09:19:31    endpoint: None
09:19:31    schema: gold
09:19:31    organization: 0
09:19:31  Registered adapter: spark=1.10.3
09:19:34    Connection test: [OK connection ok]

09:19:34  All checks pas

0

In [17]:
run_dbt("run", "--select", "stg_dbt_connection_test")

09:20:32  Running with dbt=1.11.12
09:20:33  Registered adapter: spark=1.10.3
09:20:34  Unable to do partial parsing because saved manifest not found. Starting full parse.
09:20:36  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on 'dim_time' in
package 'aws_pipeline_dev' (models/marts/dim_time.yml). Arguments to generic
tests should be nested under the `arguments` property.
09:20:36  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 1 unused configuration paths:
- models.aws_pipeline.marts
09:20:36  Found 3 models, 1 seed, 16 data tests, 517 macros
09:20:36  
09:20:36  Concurrency: 1 threads (target='dev')
09:20:36  
09:20:38  1 of 1 START sql view model gold.stg_dbt_connection_test ....................... [RUN]
09:20:38  1 of 1 OK created sql view model gold.stg_dbt_connection_test .................. [OK in 0.14s]
09:20:38  

0

In [18]:
# Terminate the EMR Serverless session when done — SPARK_REMOTE/the
# .spark_remote_env file becomes invalid, and this stops billing.
client.terminate_session(applicationId=APPLICATION_ID, sessionId=session_id)
print(f"Session {session_id} terminated.")

Session 00g78j0vi2gn3q26 terminated.
